In [9]:
# CELL 1 - Imports and API test
import requests
import pandas as pd
import time

# Test the API is reachable
response = requests.get("http://api.jolpi.ca/ergast/f1/drivers.json?limit=1000")
data = response.json()

print(response.status_code)  # Should be 200
print(data['MRData']['total'])  # Should be 879

200
879


In [10]:
# CELL 2 - Fetch all drivers who raced between 2005-2025
# Each row = one driver-season combination (514 total)
all_drivers = []

for year in range(2005, 2026):
    try:
        response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/drivers.json?limit=100", timeout=10)
        data = response.json()
        drivers = data['MRData']['DriverTable']['Drivers']
        
        for driver in drivers:
            driver['season'] = year
        
        all_drivers.extend(drivers)
    except Exception as e:
        print(f"Skipping {year}: {e}")
    
    time.sleep(0.5)

df = pd.DataFrame(all_drivers)
print(df.shape)  # Should be (514, 9)
df.head()

(514, 9)


,driverId,code,url,givenName,familyName,dateOfBirth,nationality,season,permanentNumber
0,albers,ALB,http://en.wikipedia.org/wiki/Christijan_Albers,Christijan,Albers,1979-04-16,Dutch,2005,NaN
1,alonso,ALO,http://en.wikipedia.org/wiki/Fernando_Alonso,Fernando,Alonso,1981-07-29,Spanish,2005,14
2,barrichello,BAR,http://en.wikipedia.org/wiki/Rubens_Barrichello,Rubens,Barrichello,1972-05-23,Brazilian,2005,NaN
3,button,BUT,http://en.wikipedia.org/wiki/Jenson_Button,Jenson,Button,1980-01-19,British,2005,22
4,coulthard,COU,http://en.wikipedia.org/wiki/David_Coulthard,David,Coulthard,1971-03-27,British,2005,NaN


In [11]:
# CELL 3 - Fetch all race results (2005-2025)
# Each row = one driver's result in one race
# Key columns: position, points, status (Finished/DNF), season, round
all_results = []

for year in range(2005, 2026):
    try:
        response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/results.json?limit=1000", timeout=10)
        data = response.json()
        races = data['MRData']['RaceTable']['Races']
        
        for race in races:
            round_num = race['round']
            race_name = race['raceName']
            for result in race['Results']:
                result['season'] = year
                result['round'] = round_num
                result['raceName'] = race_name
                all_results.append(result)
    except Exception as e:
        print(f"Skipping {year}: {e}")
    
    time.sleep(0.5)

df_results = pd.DataFrame(all_results)
print(df_results.shape)  # Should be (2100, 14)
df_results.head()

(2100, 14)


,number,position,positionText,points,Driver,Constructor,grid,laps,status,Time,FastestLap,season,round,raceName
0,6,1,1,10,"{'driverId': 'fisichella', 'code': 'FIS', 'url...","{'constructorId': 'renault', 'url': 'https://e...",1,57,Finished,"{'millis': '5057336', 'time': '1:24:17.336'}","{'rank': '2', 'lap': '55', 'Time': {'time': '1...",2005,1,Australian Grand Prix
1,2,2,2,8,"{'driverId': 'barrichello', 'code': 'BAR', 'ur...","{'constructorId': 'ferrari', 'url': 'https://e...",11,57,Finished,"{'millis': '5062889', 'time': '+5.553'}","{'rank': '3', 'lap': '54', 'Time': {'time': '1...",2005,1,Australian Grand Prix
2,5,3,3,6,"{'driverId': 'alonso', 'permanentNumber': '14'...","{'constructorId': 'renault', 'url': 'https://e...",13,57,Finished,"{'millis': '5064048', 'time': '+6.712'}","{'rank': '1', 'lap': '24', 'Time': {'time': '1...",2005,1,Australian Grand Prix
3,14,4,4,5,"{'driverId': 'coulthard', 'code': 'COU', 'url'...","{'constructorId': 'red_bull', 'url': 'https://...",5,57,Finished,"{'millis': '5073467', 'time': '+16.131'}","{'rank': '11', 'lap': '40', 'Time': {'time': '...",2005,1,Australian Grand Prix
4,7,5,5,4,"{'driverId': 'webber', 'code': 'WEB', 'url': '...","{'constructorId': 'williams', 'url': 'https://...",3,57,Finished,"{'millis': '5074244', 'time': '+16.908'}","{'rank': '8', 'lap': '37', 'Time': {'time': '1...",2005,1,Australian Grand Prix


In [12]:
# CELL 4 - Fetch end-of-season driver championship standings (2005-2025)
# Each row = one driver's final championship position for that season
# Used to identify WDC winners (position == 1) for filtering later
all_standings = []

for year in range(2005, 2026):
    try:
        response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/driverStandings.json?limit=100", timeout=10)
        data = response.json()
        standings_list = data['MRData']['StandingsTable']['StandingsLists']
        
        if standings_list:
            for entry in standings_list[0]['DriverStandings']:
                entry['season'] = year
                all_standings.append(entry)
    except Exception as e:
        print(f"Skipping {year}: {e}")
    
    time.sleep(0.5)

df_standings = pd.DataFrame(all_standings)
print(df_standings.shape)  # Should be (498, 7)
df_standings.head()

(498, 7)


,position,positionText,points,wins,Driver,Constructors,season
0,1,1,133,7,"{'driverId': 'alonso', 'permanentNumber': '14'...","[{'constructorId': 'renault', 'url': 'https://...",2005
1,2,2,112,7,"{'driverId': 'raikkonen', 'permanentNumber': '...","[{'constructorId': 'mclaren', 'url': 'https://...",2005
2,3,3,62,1,"{'driverId': 'michael_schumacher', 'code': 'MS...","[{'constructorId': 'ferrari', 'url': 'https://...",2005
3,4,4,60,3,"{'driverId': 'montoya', 'code': 'MON', 'url': ...","[{'constructorId': 'mclaren', 'url': 'https://...",2005
4,5,5,58,1,"{'driverId': 'fisichella', 'code': 'FIS', 'url...","[{'constructorId': 'renault', 'url': 'https://...",2005


In [13]:
# CELL 5 - Fetch end-of-season constructor championship standings (2005-2025)
# Each row = one constructor's final championship position for that season
# Used to assess how competitive a driver's car was in their peak years
all_constructor_standings = []

for year in range(2005, 2026):
    try:
        response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/constructorStandings.json?limit=100", timeout=10)
        data = response.json()
        standings_list = data['MRData']['StandingsTable']['StandingsLists']
        
        if standings_list:
            for entry in standings_list[0]['ConstructorStandings']:
                entry['season'] = year
                all_constructor_standings.append(entry)
    except Exception as e:
        print(f"Skipping {year}: {e}")
    
    time.sleep(0.5)

df_constructor_standings = pd.DataFrame(all_constructor_standings)
# Extract constructorId immediately so it's available for merging
df_constructor_standings['constructorId'] = df_constructor_standings['Constructor'].apply(lambda x: x['constructorId'])

print(df_constructor_standings.shape)  # Should be (223, 7)
df_constructor_standings.head()

(223, 7)


,position,positionText,points,wins,Constructor,season,constructorId
0,1,1,191,8,"{'constructorId': 'renault', 'url': 'https://e...",2005,renault
1,2,2,182,10,"{'constructorId': 'mclaren', 'url': 'https://e...",2005,mclaren
2,3,3,100,1,"{'constructorId': 'ferrari', 'url': 'https://e...",2005,ferrari
3,4,4,88,0,"{'constructorId': 'toyota', 'url': 'https://en...",2005,toyota
4,5,5,66,0,"{'constructorId': 'williams', 'url': 'https://...",2005,williams


In [14]:
# CELL 6 - Identify WDC winners (drivers who finished P1 in championship standings)
# Extract driverId from nested Driver dict first
df_standings['driverId'] = df_standings['Driver'].apply(lambda x: x['driverId'])

wdc_winners = df_standings[df_standings['position'] == '1']['driverId'].unique()
print(f"WDC winners in 2005-2025: {wdc_winners}")
print(f"Total: {len(wdc_winners)}")  # Should be 8

WDC winners in 2005-2025: <StringArray>
[        'alonso',      'raikkonen',       'hamilton',         'button',
         'vettel',        'rosberg', 'max_verstappen',         'norris']
Length: 8, dtype: str
Total: 8


In [15]:
# CELL 7 - Filter to non-champions and extract key IDs
# Extract driverId and constructorId from nested dicts
df_results['driverId'] = df_results['Driver'].apply(lambda x: x['driverId'])
df_results['constructorId'] = df_results['Constructor'].apply(lambda x: x['constructorId'])

non_champions = df_results[~df_results['driverId'].isin(wdc_winners)]

print(f"Total race entries: {len(df_results)}")
print(f"Non-champion race entries: {len(non_champions)}")  # Should be 1568
print(f"Unique non-champion drivers: {non_champions['driverId'].nunique()}")  # Should be 85

Total race entries: 2100
Non-champion race entries: 1568
Unique non-champion drivers: 85


In [16]:
# CELL 8 - Merge non-champions with constructor standings
# Adds constructor rank to each race result so we can measure car competitiveness
merged = non_champions.merge(
    df_constructor_standings[['constructorId', 'season', 'position']].rename(columns={'position': 'constructor_rank'}),
    on=['constructorId', 'season'],
    how='left'
)

print(merged.shape)  # Should be (1568, 17)
merged[['driverId', 'constructorId', 'season', 'position', 'constructor_rank', 'status']].head()

(1568, 17)


,driverId,constructorId,season,position,constructor_rank,status
0,fisichella,renault,2005,1,1,Finished
1,barrichello,ferrari,2005,2,3,Finished
2,coulthard,red_bull,2005,4,7,Finished
3,webber,williams,2005,5,5,Finished
4,montoya,mclaren,2005,6,2,Finished


In [17]:
# CELL 9 - Clean merged dataset: fix dtypes and handle DNFs
# Convert key columns from strings to numeric
merged['position'] = pd.to_numeric(merged['position'], errors='coerce')
merged['constructor_rank'] = pd.to_numeric(merged['constructor_rank'], errors='coerce')
merged['grid'] = pd.to_numeric(merged['grid'], errors='coerce')
merged['laps'] = pd.to_numeric(merged['laps'], errors='coerce')
merged['points'] = pd.to_numeric(merged['points'], errors='coerce')

# Flag DNFs - lapped cars ('+1 Lap' etc.) still finished, so exclude them
finished_statuses = ['Finished', '+1 Lap', '+2 Laps', '+3 Laps', '+4 Laps', '+5 Laps',
                     '+6 Laps', '+7 Laps', '+8 Laps', '+9 Laps', 'Lapped']
merged['dnf'] = ~merged['status'].isin(finished_statuses)

# Check results
print(merged.dtypes)
print(f"\nDNF count: {merged['dnf'].sum()}")
print(f"DNF rate: {merged['dnf'].mean():.1%}")  # Should be ~22%

number                  str
position              int64
positionText            str
points              float64
Driver               object
Constructor          object
grid                  int64
laps                  int64
status                  str
Time                 object
FastestLap           object
season                int64
round                   str
raceName                str
driverId                str
constructorId           str
constructor_rank      int64
dnf                    bool
dtype: object

DNF count: 355
DNF rate: 22.6%


In [18]:
# CELL 10 - Metric 1: Podium rate (podiums / races entered) for non-champions
# Exclude DNFs from race count - a DNF is not a representative race result
podium_stats = merged[merged['dnf'] == False].groupby('driverId').agg(
    races=('position', 'count'),
    podiums=('position', lambda x: (x <= 3).sum())
).reset_index()

podium_stats['podium_rate'] = podium_stats['podiums'] / podium_stats['races']

# Filter to drivers with at least 20 race finishes (removes backmarkers with lucky podiums)
podium_stats = podium_stats[podium_stats['races'] >= 20]

print(podium_stats.shape)
podium_stats.sort_values('podium_rate', ascending=False).head(10)

(21, 4)


,driverId,races,podiums,podium_rate
8,bottas,52,18,0.346154
45,leclerc,35,11,0.314286
76,trulli,22,5,0.227273
58,perez,62,13,0.209677
81,webber,35,7,0.200000
31,heidfeld,26,5,0.192308
48,massa,53,10,0.188679
68,sainz,45,7,0.155556
41,kubica,20,3,0.150000
67,russell,31,4,0.129032


In [21]:
# CELL 12 - Metric 2: Overperformer flag
# Compare driver's avg finish position vs their constructor's avg finish position
# If driver finishes better than car average = overperformer (positive delta)

finished = merged[merged['dnf'] == False].copy()

# Average finish position per driver
driver_avg = finished.groupby('driverId')['position'].mean().rename('driver_avg_pos')

# Average finish position per constructor per season
constructor_avg = finished.groupby(['constructorId', 'season'])['position'].mean().rename('constructor_avg_pos')

# Map constructor average back to each row
finished = finished.join(constructor_avg, on=['constructorId', 'season'])

# Calculate delta per driver (positive = outperforming car)
overperformer = finished.groupby('driverId').apply(
    lambda x: (x['constructor_avg_pos'] - x['position']).mean()
).reset_index()
overperformer.columns = ['driverId', 'overperformer_delta']

print(overperformer.shape)
overperformer.sort_values('overperformer_delta', ascending=False).head(10)

# Filter to drivers with at least 20 race finishes (same threshold as podium rate)
race_counts = finished.groupby('driverId')['position'].count().rename('races')
overperformer = overperformer.merge(race_counts, on='driverId')
overperformer = overperformer[overperformer['races'] >= 20]

print(overperformer.shape)
overperformer.sort_values('overperformer_delta', ascending=False).head(10)

(85, 2)
(21, 3)


,driverId,overperformer_delta,races
1,albon,1.154286,25
23,gasly,0.798851,29
37,kevin_magnussen,0.782766,35
76,trulli,0.671266,22
67,russell,0.438300,31
8,bottas,0.261218,52
31,heidfeld,0.231685,26
27,grosjean,0.208210,29
45,leclerc,0.199603,35
5,barrichello,0.199451,26


In [23]:
# CELL 13 - Define composite "successful loser" score
# Combine podium rate + overperformer delta (normalised)
from sklearn.preprocessing import MinMaxScaler

composite = podium_stats.merge(overperformer[['driverId', 'overperformer_delta']], on='driverId', how='inner')

scaler = MinMaxScaler()
composite[['podium_rate_scaled', 'overperformer_scaled']] = scaler.fit_transform(
    composite[['podium_rate', 'overperformer_delta']]
)

composite['composite_score'] = (composite['podium_rate_scaled'] + composite['overperformer_scaled']) / 2

print(composite.shape)
composite.sort_values('composite_score', ascending=False).head(10)

(21, 8)


,driverId,races,podiums,podium_rate,overperformer_delta,podium_rate_scaled,overperformer_scaled,composite_score
2,bottas,52,18,0.346154,0.261218,1.000000,0.515175,0.757587
18,trulli,22,5,0.227273,0.671266,0.656566,0.737780,0.697173
10,leclerc,35,11,0.314286,0.199603,0.907937,0.481726,0.694831
5,heidfeld,26,5,0.192308,0.231685,0.555556,0.499142,0.527349
20,webber,35,7,0.200000,0.147619,0.577778,0.453505,0.515641
13,perez,62,13,0.209677,0.047357,0.605735,0.399075,0.502405
0,albon,25,0,0.000000,1.154286,0.000000,1.000000,0.500000
15,russell,31,4,0.129032,0.438300,0.372760,0.611308,0.492034
7,kevin_magnussen,35,1,0.028571,0.782766,0.082540,0.798311,0.440425
11,massa,53,10,0.188679,-0.149072,0.545073,0.292438,0.418756


In [24]:
# CELL 14 - Fetch qualifying data (2005-2025)
# Needed for teammate delta: qualifying gap + race position gap vs teammate
all_qualifying = []

for year in range(2005, 2026):
    try:
        response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/qualifying.json?limit=1000", timeout=10)
        data = response.json()
        races = data['MRData']['RaceTable']['Races']
        
        for race in races:
            round_num = race['round']
            race_name = race['raceName']
            for result in race['QualifyingResults']:
                result['season'] = year
                result['round'] = round_num
                result['raceName'] = race_name
                all_qualifying.append(result)
    except Exception as e:
        print(f"Skipping {year}: {e}")
    
    time.sleep(0.5)

df_qualifying = pd.DataFrame(all_qualifying)
print(df_qualifying.shape)
df_qualifying.head()

(2100, 10)


,number,position,Driver,Constructor,Q1,Q2,season,round,raceName,Q3
0,6,1,"{'driverId': 'fisichella', 'code': 'FIS', 'url...","{'constructorId': 'renault', 'url': 'https://e...",1:33.171,1:28.289,2005,1,Australian Grand Prix,NaN
1,16,2,"{'driverId': 'trulli', 'code': 'TRU', 'url': '...","{'constructorId': 'toyota', 'url': 'https://en...",1:35.270,1:29.159,2005,1,Australian Grand Prix,NaN
2,7,3,"{'driverId': 'webber', 'code': 'WEB', 'url': '...","{'constructorId': 'williams', 'url': 'https://...",1:36.717,1:28.279,2005,1,Australian Grand Prix,NaN
3,11,4,"{'driverId': 'villeneuve', 'code': 'VIL', 'url...","{'constructorId': 'sauber', 'url': 'https://en...",1:36.984,1:29.862,2005,1,Australian Grand Prix,NaN
4,14,5,"{'driverId': 'coulthard', 'code': 'COU', 'url'...","{'constructorId': 'red_bull', 'url': 'https://...",1:38.320,1:28.892,2005,1,Australian Grand Prix,NaN


In [25]:
# CELL 15 - Calculate teammate delta score
# Extract driverId and constructorId from nested dicts
df_qualifying['driverId'] = df_qualifying['Driver'].apply(lambda x: x['driverId'])
df_qualifying['constructorId'] = df_qualifying['Constructor'].apply(lambda x: x['constructorId'])
df_qualifying['position'] = pd.to_numeric(df_qualifying['position'], errors='coerce')

# Filter to non-champions only
qual_non_champ = df_qualifying[~df_qualifying['driverId'].isin(wdc_winners)]

# For each race, pair teammates (same constructor, same race, same season)
teammate_pairs = qual_non_champ.merge(
    qual_non_champ[['driverId', 'constructorId', 'season', 'round', 'position']],
    on=['constructorId', 'season', 'round'],
    suffixes=('_driver', '_teammate')
)

# Remove self-comparisons
teammate_pairs = teammate_pairs[teammate_pairs['driverId_driver'] != teammate_pairs['driverId_teammate']]

# Qualifying delta: positive = driver qualified ahead of teammate
teammate_pairs['qual_delta'] = teammate_pairs['position_teammate'] - teammate_pairs['position_driver']

# Aggregate into career score per driver
teammate_delta = teammate_pairs.groupby('driverId_driver').agg(
    qual_delta_avg=('qual_delta', 'mean'),
    comparisons=('qual_delta', 'count')
).reset_index().rename(columns={'driverId_driver': 'driverId'})

# Filter to minimum 20 comparisons
teammate_delta = teammate_delta[teammate_delta['comparisons'] >= 20]

print(teammate_delta.shape)
teammate_delta.sort_values('qual_delta_avg', ascending=False).head(10)

(23, 3)


,driverId,qual_delta_avg,comparisons
63,russell,3.300000,20
23,gasly,2.914286,35
61,ricciardo,2.878788,33
1,albon,2.347826,23
8,bottas,2.090909,33
72,trulli,2.000000,31
41,kubica,1.833333,24
45,leclerc,1.760000,25
32,hulkenberg,1.180328,61
27,grosjean,0.666667,33


In [26]:
# CELL 16 - Final composite score (podium rate + overperformer + teammate delta)
composite_final = podium_stats.merge(
    overperformer[['driverId', 'overperformer_delta']], on='driverId', how='inner'
).merge(
    teammate_delta[['driverId', 'qual_delta_avg']], on='driverId', how='inner'
)

scaler = MinMaxScaler()
composite_final[['podium_rate_scaled', 'overperformer_scaled', 'teammate_scaled']] = scaler.fit_transform(
    composite_final[['podium_rate', 'overperformer_delta', 'qual_delta_avg']]
)

composite_final['composite_score'] = (
    composite_final['podium_rate_scaled'] + 
    composite_final['overperformer_scaled'] + 
    composite_final['teammate_scaled']
) / 3

print(composite_final.shape)
composite_final.sort_values('composite_score', ascending=False).head(15)

(19, 10)


,driverId,races,podiums,podium_rate,overperformer_delta,qual_delta_avg,podium_rate_scaled,overperformer_scaled,teammate_scaled,composite_score
1,bottas,52,18,0.346154,0.261218,2.090909,1.000000,0.515175,0.805797,0.773657
17,trulli,22,5,0.227273,0.671266,2.000000,0.656566,0.737780,0.791196,0.728514
9,leclerc,35,11,0.314286,0.199603,1.760000,0.907937,0.481726,0.752647,0.714103
14,russell,31,4,0.129032,0.438300,3.300000,0.372760,0.611308,1.000000,0.661356
0,albon,25,0,0.000000,1.154286,2.347826,0.000000,1.000000,0.847063,0.615688
2,gasly,29,0,0.000000,0.798851,2.914286,0.000000,0.807043,0.938047,0.581697
13,ricciardo,44,3,0.068182,0.153084,2.878788,0.196970,0.456472,0.932345,0.528596
12,perez,62,13,0.209677,0.047357,0.250000,0.605735,0.399075,0.510113,0.504974
6,kevin_magnussen,35,1,0.028571,0.782766,0.358974,0.082540,0.798311,0.527616,0.469489
10,massa,53,10,0.188679,-0.149072,0.000000,0.545073,0.292438,0.469958,0.435823


In [27]:
# CELL 17 - Export all datasets to CSV
merged.to_csv('f1_merged_clean.csv', index=False)
podium_stats.to_csv('f1_podium_stats.csv', index=False)
overperformer.to_csv('f1_overperformer.csv', index=False)
teammate_delta.to_csv('f1_teammate_delta.csv', index=False)
composite_final.to_csv('f1_composite_final.csv', index=False)

print("All files saved!")

All files saved!
